In [5]:
# You only need to run this block once per session to install Gurobi (and other libraries)

%pip install pyomo gurobipy pandas

In [ ]:
# v0.2 - 9/3/26

import gurobipy as gp
from gurobipy import GRB, Model

import pandas as pd
from typing import Any
import json

from datetime import datetime, date as Date
import calendar
from IPython.display import HTML

# ============= PARAMETERS =============
YEAR = 2026
MONTH = 1

residents = [
    "A",
    "B",
    "C",
    "D",
    "E",
    "F",
    "G",
]

vacation_requests = {
    "A": ["2026-01-03", "2026-01-14", "2026-01-24"],
    "B": ["2026-01-03", "2026-01-15", "2026-01-25"],
    "C": ["2026-01-03", "2026-01-16", "2026-01-26"],
    "D": ["2026-01-03", "2026-01-17", "2026-01-27"],
    "E": ["2026-01-03", "2026-01-18", "2026-01-28"],
    "F": ["2026-01-03", "2026-01-19", "2026-01-29"],
    "G": ["2026-01-03", "2026-01-20", "2026-01-30"],
}


# ============= GENERATE =============
first_date = Date(YEAR, MONTH, 1)
last_day = calendar.monthrange(YEAR, MONTH)[1]
last_date = Date(YEAR, MONTH, last_day)
dates = (
    pd.date_range(start=first_date, end=last_date)
    .strftime("%Y-%m-%d")
    .tolist()
)


# ============= MODEL PARAMETERS =============
m: Model = Model()
# m.setParam('LogToConsole', 0)


# ============= BUILD =============
# Decision variables
x_rd = m.addVars(len(residents), len(dates), 
                    lb=0, ub=1, 
                    vtype=GRB.BINARY)

# Set variable names
for (r, resident) in enumerate(residents):
    for (d, date) in enumerate(dates):
        x_rd[r, d].VarName = f"{resident}_{date}"

# One res for each date
for (d, date) in enumerate(dates):
    m.addConstr(gp.quicksum(x_rd[r, d] for (r, resident) in enumerate(residents)) == 1,
                name=f"single_shift_{date}")

# Resident coverage ub of 7 across the entire month
for (r, resident) in enumerate(residents):
    m.addConstr(gp.quicksum(x_rd[r, d] for (d, date) in enumerate(dates)) <= 7,
                name=f"coverage_ub_{resident}")
        
# Obj fxn: minimize number of assignments to resident 0 (i.e., the first resident, "A")
# m.setObjective(
#     gp.quicksum(x_rd[r, d] for r, d in x_rd if r == 0),
#     GRB.MINIMIZE
# )

# Obj fxn: minimize number of assignments on vacation days
m.setObjective(
    gp.quicksum(
        x_rd[r, d]
        for r, resident in enumerate(residents)
        for d, shift_date in enumerate(dates)
        if shift_date in vacation_requests[resident]
    ),
    GRB.MINIMIZE
)


# ============= SOLVE =============
m.optimize()
tolerance = 0.5

# Extract solution variable as boolean
x_rd_sol = [[x_rd[i, j].X > tolerance for j in range(len(dates))]
          for i in range(len(residents))]

# ============= REPORT =============
# print vars
# for v in m.getVars():
#     print(f"{v.VarName} = {v.X}")

# infeasible: bool = m.Status == GRB.INFEASIBLE
# print(m.Status)

# ---------- Table ----------
table_rows = []

for r, resident in enumerate(residents):
    total_shifts = sum(
        x_rd_sol[r][d]
        for d in range(num_dates)
    )

    weekend_shifts = sum(
        x_rd_sol[r][d]
        for d, shift_date in enumerate(dates)
        if datetime.strptime(shift_date, "%Y-%m-%d").weekday() >= 5  # Saturday (5) or Sunday (6)
    )

    denied_requests = sum(
        x_rd_sol[r][d]
        for d, shift_date in enumerate(dates)
        if shift_date in vacation_requests[resident]
    )

    requested_vacations = len(vacation_requests[resident])

    table_rows.append({
        "Resident": resident,
        "Total Shifts": total_shifts,
        "Weekend Shifts": weekend_shifts,
        "Vacation Requests": requested_vacations,
        "Requests Denied": denied_requests,
    })

table_rows_df = pd.DataFrame(table_rows)
display(table_rows_df)

# ---------- Calendar ----------
class Schedule(calendar.HTMLCalendar):
    def __init__(self, event_names, dates, occurs):
        # initialize calendar.HTMLCalendar to start weeks on Sunday
        super().__init__(calendar.SUNDAY)

        self.assignments = {}
        for r in range(len(residents)):
            for d in range(len(dates)):
                if (x_rd_sol[r][d]):
                    self.assignments[dates[d]] = residents[r]
        

    def formatday(self, day, weekday):
        if day == 0:
            # empty table cell
            return '<td></td>'

        # render cell with day and assignment
        event_date = Date(
            self.current_year,
            self.current_month,
            day
        ).strftime("%Y-%m-%d")
        event = self.assignments.get(event_date, "")

        event_html = (f'<div>{event}</div>') if event else ""
        return (f'<td style="border:1px solid #ffffff; width:80px; height:60px; vertical-align:top">'
                f'<strong>{day}</strong>'
                f'{event_html}'
                f'</td>')

    def formatmonth(self, theyear, themonth, withyear=True):
        self.current_year = theyear
        self.current_month = themonth
        return super().formatmonth(theyear, themonth, withyear)

schedule = Schedule(residents, dates, x_rd_sol)
HTML(schedule.formatmonth(YEAR, MONTH))



Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 38 rows, 217 columns and 434 nonzeros (Min)
Model fingerprint: 0x8796dc9b
Model has 21 linear objective coefficients
Variable types: 0 continuous, 217 integer (217 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 7e+00]

Found heuristic solution: objective 4.0000000
Presolve time: 0.00s
Presolved: 38 rows, 217 columns, 434 nonzeros
Variable types: 0 continuous, 217 integer (217 binary)

Root relaxation: objective 1.000000e+00, 35 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

* 

,Resident,Total Shifts,Weekend Shifts,Vacation Requests,Requests Denied
0,TEST,7,1,3,0
1,B,2,1,3,0
2,C,3,1,3,0
3,D,6,3,3,1
4,E,7,3,3,0
5,F,4,0,3,0
6,G,2,0,3,0
